# Практическая работа № 5 — Классификатор и анализ метрик (уровень B)

**Студент:** Barsbek Konurbaev

Цель: построить бинарный классификатор, оценить ROC/PR-кривые и подобрать порог с учётом разной стоимости ошибок.

## 1. Реальный датасет и постановка задачи
Используется Breast Cancer Wisconsin (Diagnostic) из scikit-learn: 569 реальных наблюдений, 30 числовых признаков, два класса. Для демонстрации цены ошибки считаем ложноотрицательный результат для класса malignant более дорогим, чем ложноположительный.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score,
    accuracy_score
)

data = load_breast_cancer(as_frame=True)
X = data.data
# Перекодируем malignant как положительный класс 1
y = (data.target == 0).astype(int)
print("Размер:", X.shape)
print("Классы: 1 = malignant, 0 = benign")
print(y.value_counts())

## 2. Обучение модели и получение вероятностей

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, proba)
ap = average_precision_score(y_test, proba)
print("ROC-AUC:", round(roc_auc, 4))
print("Average Precision (PR):", round(ap, 4))

## 3. ROC и PR-кривые

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, proba)
precision, recall, pr_thresholds = precision_recall_curve(y_test, proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.title("ROC-кривая")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"Average Precision = {ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR-кривая")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Подбор порога с учётом цены ошибки
Используем заданное учебное соотношение стоимости: FN = 5 условных единиц, FP = 1. Выбираем порог, минимизирующий суммарную стоимость ошибок на тестовой выборке для демонстрации; в реальном проекте порог следует выбирать на отдельной validation-выборке, не на test.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
FN_COST = 5
FP_COST = 1
rows = []

for threshold in thresholds:
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    rows.append({
        "threshold": threshold,
        "cost": FN_COST * fn + FP_COST * fp,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "FP": fp, "FN": fn
    })

threshold_table = pd.DataFrame(rows)
best_row = threshold_table.loc[threshold_table["cost"].idxmin()]
display(pd.DataFrame([
    {"threshold": 0.5, **threshold_table.iloc[(threshold_table["threshold"]-0.5).abs().argmin()].drop("threshold").to_dict()},
    best_row.to_dict()
], index=["Default 0.5", "Cost-minimizing threshold"]).round(4))
print("Подобранный порог:", round(float(best_row["threshold"]), 3))

## 5. Сравнение матриц ошибок

In [ ]:
chosen_threshold = float(best_row["threshold"])
pred_default = (proba >= 0.5).astype(int)
pred_tuned = (proba >= chosen_threshold).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred_default, display_labels=["benign", "malignant"], ax=axes[0], colorbar=False)
axes[0].set_title("Порог 0.5")
ConfusionMatrixDisplay.from_predictions(y_test, pred_tuned, display_labels=["benign", "malignant"], ax=axes[1], colorbar=False)
axes[1].set_title(f"Порог {chosen_threshold:.2f}")
plt.tight_layout()
plt.show()

display(pd.DataFrame([
    {"threshold": 0.5, "precision": precision_score(y_test, pred_default, zero_division=0),
     "recall": recall_score(y_test, pred_default, zero_division=0),
     "f1": f1_score(y_test, pred_default, zero_division=0)},
    {"threshold": chosen_threshold, "precision": precision_score(y_test, pred_tuned, zero_division=0),
     "recall": recall_score(y_test, pred_tuned, zero_division=0),
     "f1": f1_score(y_test, pred_tuned, zero_division=0)}
], index=["Default", "Adjusted"]).round(4))

## 6. Вывод
Объясните, как изменились FN и FP после изменения порога. При высокой цене FN обычно важнее не пропустить положительный класс, но это может увеличить FP. ROC-AUC оценивает ранжирование вероятностей по разным порогам; он не выбирает порог сам по себе.